In [838]:
# Sqlite setup

# Import sqlite libraries
import sqlite3
import pandas as pd

# Create a new sqlite database in memory
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

**Database Schema**

In [839]:
# Departments table

cursor.execute("""
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT
)
""")

| Column Name    | Data Type | Description            |
| -------------- | --------- | ---------------------- |
| DepartmentID   | INTEGER   | Primary Key            |
| DepartmentName | TEXT      | Name of the department |


In [840]:
# Rootcauses table

cursor.execute("""
CREATE TABLE RootCauses (
    RootCauseID INTEGER PRIMARY KEY,
    RootCause TEXT
)
""")

| Column Name | Data Type | Description           |
| ----------- | --------- | --------------------- |
| RootCauseID | INTEGER   | Primary Key           |
| RootCause   | TEXT      | Reason for the ticket |


In [841]:
# IT_Tickets table

cursor.execute("""
CREATE TABLE IT_Tickets (
    TicketID INTEGER PRIMARY KEY,
    DepartmentID INTEGER,
    RootCauseID INTEGER,
    Status TEXT,
    Priority TEXT,
    ResolutionHours INTEGER,
    CreatedDate TEXT,
    FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID),
    FOREIGN KEY (RootCauseID) REFERENCES RootCauses(RootCauseID)
)
""")

| Column Name     | Data Type | Description                         |
| --------------- | --------- | ----------------------------------- |
| TicketID        | INTEGER   | Primary Key                         |
| DepartmentID    | INTEGER   | Foreign Key referencing Departments |
| RootCauseID     | INTEGER   | Foreign Key referencing RootCauses  |
| Status          | TEXT      | Ticket status: Open, Closed         |
| Priority        | TEXT      | Priority level: High, Medium, Low   |
| ResolutionHours | INTEGER   | Hours taken to resolve the ticket   |
| CreatedDate     | TEXT      | Ticket creation date (YYYY-MM-DD)   |


**Insert data into table**

In [842]:
# Insert data into Departments

departments = [
    (1,'IT'),
    (2,'HR'),
    (3,'Finance'),
    (4,'Operations'),
    (5,'Customer Service'),

]

conn.executemany("INSERT INTO Departments VALUES (?,?)", departments)
conn.commit()

# Verify
import pandas as pd
pd.read_sql("SELECT * FROM Departments", conn)

,DepartmentID,DepartmentName
0,1,IT
1,2,HR
2,3,Finance
3,4,Operations
4,5,Customer Service


In [843]:
# Insert data into RootCauses

root_causes = [
    (1,'Configuration'),
    (2,'Software Failure'),
    (3,'Network Issue'),
    (4,'Hardware Failure'),
    (5,'Security')
]

conn.executemany("INSERT INTO RootCauses VALUES (?,?)", root_causes)
conn.commit()

# Verify
pd.read_sql("SELECT * FROM RootCauses", conn)

,RootCauseID,RootCause
0,1,Configuration
1,2,Software Failure
2,3,Network Issue
3,4,Hardware Failure
4,5,Security


In [844]:
# Insert data into IT_Tickets

tickets = [
    (1,1,1,'Closed','High',5,'2026-03-01'),
    (2,2,5,'Closed','Medium',8,'2026-03-02'),
    (3,1,3,'Open','Low',12,'2026-03-03'),
    (4,5,1,'Closed','High',3,'2026-03-04'),
    (5,4,4,'Open','Medium',7,'2026-03-05'),
    (6,2,2,'Closed','Low',4,'2026-03-06'),
    (7,3,3,'Closed','High',6,'2026-03-07'),
    (8,1,1,'Open','High',9,'2026-03-08'),
    (9,4,2,'Closed','Medium',2,'2026-03-09'),
    (10,5,4,'Open','Low',10,'2026-03-10'),
    (11,5,2,'Closed','Low',4,'2026-01-04'),
    (12,3,3,'Closed','High',6,'2026-01-11'),
    (13,1,1,'Open','Low',9,'2026-03-01'),
    (14,4,5,'Closed','Medium',2,'2026-01-03'),
    (15,5,4,'Open','Low',10,'2026-01-11')
]

conn.executemany("INSERT INTO IT_Tickets VALUES (?,?,?,?,?,?,?)", tickets)
conn.commit()

# Verify
pd.read_sql("SELECT * FROM IT_Tickets", conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,1,1,1,Closed,High,5,2026-03-01
1,2,2,5,Closed,Medium,8,2026-03-02
2,3,1,3,Open,Low,12,2026-03-03
3,4,5,1,Closed,High,3,2026-03-04
4,5,4,4,Open,Medium,7,2026-03-05
5,6,2,2,Closed,Low,4,2026-03-06
6,7,3,3,Closed,High,6,2026-03-07
7,8,1,1,Open,High,9,2026-03-08
8,9,4,2,Closed,Medium,2,2026-03-09
9,10,5,4,Open,Low,10,2026-03-10


**Queries**

In [845]:
# Count the total number of tickets

query = """
SELECT COUNT(*) AS TotalTickets
FROM IT_Tickets
"""
pd.read_sql(query, conn)

,TotalTickets
0,15


In [846]:
# Find all columns from the IT_Tickets table

query = """
SELECT *
FROM IT_Tickets
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,1,1,1,Closed,High,5,2026-03-01
1,2,2,5,Closed,Medium,8,2026-03-02
2,3,1,3,Open,Low,12,2026-03-03
3,4,5,1,Closed,High,3,2026-03-04
4,5,4,4,Open,Medium,7,2026-03-05
5,6,2,2,Closed,Low,4,2026-03-06
6,7,3,3,Closed,High,6,2026-03-07
7,8,1,1,Open,High,9,2026-03-08
8,9,4,2,Closed,Medium,2,2026-03-09
9,10,5,4,Open,Low,10,2026-03-10


In [847]:
# List ticket ID, department name, and root cause

query = """
SELECT t.TicketID, d.DepartmentName, r.RootCause
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID;
"""
pd.read_sql(query, conn)

,TicketID,DepartmentName,RootCause
0,1,IT,Configuration
1,2,HR,Security
2,3,IT,Network Issue
3,4,Customer Service,Configuration
4,5,Operations,Hardware Failure
5,6,HR,Software Failure
6,7,Finance,Network Issue
7,8,IT,Configuration
8,9,Operations,Software Failure
9,10,Customer Service,Hardware Failure


In [848]:
# Find all tickets with Status = 'Open'

query = """
SELECT *
FROM IT_Tickets
WHERE status = 'Open';
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,3,1,3,Open,Low,12,2026-03-03
1,5,4,4,Open,Medium,7,2026-03-05
2,8,1,1,Open,High,9,2026-03-08
3,10,5,4,Open,Low,10,2026-03-10
4,13,1,1,Open,Low,9,2026-03-01
5,15,5,4,Open,Low,10,2026-01-11


In [849]:
# Display only TicketID, DepartmentID, and Priority

query = """
SELECT TicketID, DepartmentID, Priority
FROM IT_Tickets;
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,Priority
0,1,1,High
1,2,2,Medium
2,3,1,Low
3,4,5,High
4,5,4,Medium
5,6,2,Low
6,7,3,High
7,8,1,High
8,9,4,Medium
9,10,5,Low


In [850]:
# Count tickets per department

query = """
SELECT d.DepartmentName, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
GROUP BY d.DepartmentName;
"""
pd.read_sql(query, conn)

,DepartmentName,TicketCount
0,Customer Service,4
1,Finance,2
2,HR,2
3,IT,4
4,Operations,3


In [851]:
# Count tickets per root cause

query = """
SELECT r.RootCause, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
GROUP BY r.RootCause
"""
pd.read_sql(query, conn)

,RootCause,TicketCount
0,Configuration,4
1,Hardware Failure,3
2,Network Issue,3
3,Security,2
4,Software Failure,3


In [852]:
# Departments with more than 3 tickets

query = """
SELECT d.DepartmentName, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
GROUP BY d.DepartmentName
HAVING COUNT(*) > 3;
"""
pd.read_sql(query, conn)

,DepartmentName,TicketCount
0,Customer Service,4
1,IT,4


In [853]:
# Find all tickets with ResolutionHours > 5

query = """
SELECT *
FROM IT_Tickets
WHERE ResolutionHours > 5;
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,2,2,5,Closed,Medium,8,2026-03-02
1,3,1,3,Open,Low,12,2026-03-03
2,5,4,4,Open,Medium,7,2026-03-05
3,7,3,3,Closed,High,6,2026-03-07
4,8,1,1,Open,High,9,2026-03-08
5,10,5,4,Open,Low,10,2026-03-10
6,12,3,3,Closed,High,6,2026-01-11
7,13,1,1,Open,Low,9,2026-03-01
8,15,5,4,Open,Low,10,2026-01-11


In [854]:
# Average resolution hours per department

query = """
SELECT d.DepartmentName, AVG(ResolutionHours) AS AverageHours
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
GROUP BY d.DepartmentName;
"""
pd.read_sql(query, conn)

,DepartmentName,AverageHours
0,Customer Service,6.750000
1,Finance,6.000000
2,HR,6.000000
3,IT,8.750000
4,Operations,3.666667


In [855]:
# Average ResolutionHours of all tickets

query = """
SELECT AVG(ResolutionHours) AS AverageResolutionHours
FROM IT_Tickets;
"""
pd.read_sql(query, conn)

,AverageResolutionHours
0,6.466667


In [856]:
# Average resolution hours for closed tickets only

query = """
SELECT AVG(ResolutionHours) AS AverageClosedHours
FROM IT_Tickets
WHERE Status = 'Closed';
"""
pd.read_sql(query, conn)

,AverageClosedHours
0,4.444444


In [857]:
# Average resolution hours per department for closed tickets only

query = """
SELECT d.DepartmentName, AVG(ResolutionHours) AS AvgClosedHours
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
WHERE Status='Closed'
GROUP BY d.DepartmentName
"""
pd.read_sql(query, conn)

,DepartmentName,AvgClosedHours
0,Customer Service,3.5
1,Finance,6.0
2,HR,6.0
3,IT,5.0
4,Operations,2.0


In [858]:
# Find tickets with resolution hours above the average

query = """
SELECT *
FROM IT_Tickets
WHERE ResolutionHours > (SELECT AVG(ResolutionHours)
FROM IT_Tickets);
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,2,2,5,Closed,Medium,8,2026-03-02
1,3,1,3,Open,Low,12,2026-03-03
2,5,4,4,Open,Medium,7,2026-03-05
3,8,1,1,Open,High,9,2026-03-08
4,10,5,4,Open,Low,10,2026-03-10
5,13,1,1,Open,Low,9,2026-03-01
6,15,5,4,Open,Low,10,2026-01-11


In [859]:
# Department with longest average resolution time for closed tickets

query = """
SELECT d.DepartmentName, AVG(ResolutionHours) AS AverageClosedHours
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
WHERE t.Status='Closed'
GROUP BY d.DepartmentName
ORDER BY AverageClosedHours DESC
LIMIT 1
"""
pd.read_sql(query, conn)

,DepartmentName,AverageClosedHours
0,HR,6.0


In [860]:
# High-priority tickets per department

query = """
SELECT d.DepartmentName, COUNT(*) AS HighPriorityTickets
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
WHERE Priority='High'
GROUP BY d.DepartmentName
"""
pd.read_sql(query, conn)

,DepartmentName,HighPriorityTickets
0,Customer Service,1
1,Finance,2
2,IT,2


In [861]:
# Tickets count by department and root cause for closed tickets

query = """
SELECT d.DepartmentName, r.RootCause, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
WHERE Status='Closed'
GROUP BY d.DepartmentName, r.RootCause
"""
pd.read_sql(query, conn)

,DepartmentName,RootCause,TicketCount
0,Customer Service,Configuration,1
1,Customer Service,Software Failure,1
2,Finance,Network Issue,2
3,HR,Security,1
4,HR,Software Failure,1
5,IT,Configuration,1
6,Operations,Security,1
7,Operations,Software Failure,1


In [862]:
# Count tickets by department and root cause for closed high-priority tickets

query = """
SELECT d.DepartmentName, r.RootCause, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
WHERE Status='Closed' AND Priority='High'
GROUP BY d.DepartmentName, r.RootCause;
"""
pd.read_sql(query, conn)

,DepartmentName,RootCause,TicketCount
0,Customer Service,Configuration,1
1,Finance,Network Issue,2
2,IT,Configuration,1


In [863]:
#  Open high-priority tickets with department and root cause

query = """
SELECT t.TicketID, d.DepartmentName, r.RootCause, t.ResolutionHours
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
WHERE t.Status='Open' AND t.Priority='High'
"""
pd.read_sql(query, conn)

,TicketID,DepartmentName,RootCause,ResolutionHours
0,8,IT,Configuration,9


In [864]:
# List all tickets sorted by Priority (High to Low)

query = """
SELECT *
FROM IT_Tickets
ORDER BY CASE Priority
    WHEN 'High' THEN 1
    WHEN 'Medium' THEN 2
    WHEN 'Low' THEN 3
END;
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,1,1,1,Closed,High,5,2026-03-01
1,4,5,1,Closed,High,3,2026-03-04
2,7,3,3,Closed,High,6,2026-03-07
3,8,1,1,Open,High,9,2026-03-08
4,12,3,3,Closed,High,6,2026-01-11
5,2,2,5,Closed,Medium,8,2026-03-02
6,5,4,4,Open,Medium,7,2026-03-05
7,9,4,2,Closed,Medium,2,2026-03-09
8,14,4,5,Closed,Medium,2,2026-01-03
9,3,1,3,Open,Low,12,2026-03-03


In [865]:
# Percentage of tickets per Priority

query = """
SELECT Priority, ROUND(100.0*COUNT(*)/(SELECT COUNT(*)
FROM IT_Tickets),2) AS Percentage
FROM IT_Tickets
GROUP BY Priority
"""
pd.read_sql(query, conn)

,Priority,Percentage
0,High,33.33
1,Low,40.00
2,Medium,26.67


In [866]:
# Add urgency column based on resolution hours

query = """
SELECT TicketID, ResolutionHours,
CASE
    WHEN ResolutionHours < 5 THEN 'Low'
    WHEN ResolutionHours BETWEEN 5 AND 7 THEN 'Medium'
    ELSE 'High'
END AS Urgency
FROM IT_Tickets;
"""
pd.read_sql(query, conn)

,TicketID,ResolutionHours,Urgency
0,1,5,Medium
1,2,8,High
2,3,12,High
3,4,3,Low
4,5,7,Medium
5,6,4,Low
6,7,6,Medium
7,8,9,High
8,9,2,Low
9,10,10,High


In [867]:
# List tickets sorted by CreatedDate ascending

query = """
SELECT *
FROM IT_Tickets
ORDER BY CreatedDate ASC;
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,14,4,5,Closed,Medium,2,2026-01-03
1,11,5,2,Closed,Low,4,2026-01-04
2,12,3,3,Closed,High,6,2026-01-11
3,15,5,4,Open,Low,10,2026-01-11
4,1,1,1,Closed,High,5,2026-03-01
5,13,1,1,Open,Low,9,2026-03-01
6,2,2,5,Closed,Medium,8,2026-03-02
7,3,1,3,Open,Low,12,2026-03-03
8,4,5,1,Closed,High,3,2026-03-04
9,5,4,4,Open,Medium,7,2026-03-05


In [868]:
# Find the maximum and minimum ResolutionHours

query = """
SELECT
    MAX(ResolutionHours) AS MaximumHours,
    MIN(ResolutionHours) AS MinimumHours
FROM IT_Tickets;
"""
pd.read_sql(query, conn)

,MaximumHours,MinimumHours
0,12,2


In [869]:
# List all unique Status values in the IT_Tickets table

query = """
SELECT DISTINCT Status
FROM IT_Tickets;
"""
pd.read_sql(query, conn)

,Status
0,Closed
1,Open


In [870]:
# Join with Departments

query = """
SELECT t.TicketID, d.DepartmentName
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
"""
pd.read_sql(query, conn)

,TicketID,DepartmentName
0,1,IT
1,2,HR
2,3,IT
3,4,Customer Service
4,5,Operations
5,6,HR
6,7,Finance
7,8,IT
8,9,Operations
9,10,Customer Service


In [871]:
# Join with RootCauses

query = """
SELECT t.TicketID, r.RootCause
FROM IT_Tickets t
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
"""
pd.read_sql(query, conn)

,TicketID,RootCause
0,1,Configuration
1,2,Security
2,3,Network Issue
3,4,Configuration
4,5,Hardware Failure
5,6,Software Failure
6,7,Network Issue
7,8,Configuration
8,9,Software Failure
9,10,Hardware Failure


In [872]:
# Open tickets created in last 7 days

query = """
SELECT * FROM IT_Tickets
WHERE Status='Open' AND date(CreatedDate) >= date('2026-01-10','-7 days')
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,3,1,3,Open,Low,12,2026-03-03
1,5,4,4,Open,Medium,7,2026-03-05
2,8,1,1,Open,High,9,2026-03-08
3,10,5,4,Open,Low,10,2026-03-10
4,13,1,1,Open,Low,9,2026-03-01
5,15,5,4,Open,Low,10,2026-01-11


In [873]:
# Department with highest number of open tickets

query = """
SELECT d.DepartmentName, COUNT(*) AS OpenTickets
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
WHERE Status='Open'
GROUP BY d.DepartmentName
ORDER BY OpenTickets DESC
LIMIT 1
"""
pd.read_sql(query, conn)

,DepartmentName,OpenTickets
0,IT,3


In [874]:
# Top 3 tickets by ResolutionHours

query = """
SELECT * FROM IT_Tickets
ORDER BY ResolutionHours DESC
LIMIT 3
"""
pd.read_sql(query, conn)

,TicketID,DepartmentID,RootCauseID,Status,Priority,ResolutionHours,CreatedDate
0,3,1,3,Open,Low,12,2026-03-03
1,10,5,4,Open,Low,10,2026-03-10
2,15,5,4,Open,Low,10,2026-01-11


In [875]:
# Total resolution hours grouped by department and root cause

query = """
SELECT d.DepartmentName, r.RootCause, SUM(ResolutionHours) AS TotalHours
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
GROUP BY d.DepartmentName, r.RootCause
"""
pd.read_sql(query, conn)

,DepartmentName,RootCause,TotalHours
0,Customer Service,Configuration,3
1,Customer Service,Hardware Failure,20
2,Customer Service,Software Failure,4
3,Finance,Network Issue,12
4,HR,Security,8
5,HR,Software Failure,4
6,IT,Configuration,23
7,IT,Network Issue,12
8,Operations,Hardware Failure,7
9,Operations,Security,2


In [876]:
# Tickets per department excluding 'Configuration'

query = """
SELECT d.DepartmentName, COUNT(*) AS TicketCount
FROM IT_Tickets t
JOIN Departments d ON t.DepartmentID = d.DepartmentID
JOIN RootCauses r ON t.RootCauseID = r.RootCauseID
WHERE r.RootCause != 'Configuration'
GROUP BY d.DepartmentName
"""
pd.read_sql(query, conn)

,DepartmentName,TicketCount
0,Customer Service,3
1,Finance,2
2,HR,2
3,IT,1
4,Operations,3
